In [1]:
# imports and settings
import os
import shutil
import importlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda
import seaborn as sns
from MDAnalysis.analysis.pca import PCA         # for PCA
from IPython.display import display             # for data frame display
from multiprocessing import Pool                # for multiprocessing
from tqdm import tqdm                           # for progress bars
import gc

# SETTINGS #

# base directories
base_directory = "/biggin/b212/bioc1781/Projects/CTNS/human/monomer/red-msa/with-dropout/"
structure_directory = base_directory + "ensemble/" # directory with the AF2 ensemble, cannot be the same as base directory

# list of msa depths used (for file naming)
msa_depths = ['8-16', '16-32', '32-64', '64-128', '128-256'] # , '256-512', 512-1024'

# threshold for discarding structures from ensemble
thresh_pLDDT = 85 #93 #85

# residue selections for rmsd and pca calculations
conserved_residues = ""
ignored_residues = ""
always_include = ""

# optional: define some useful MDA selection tokens selections here
IF_ref1 = ""
IF_ref2 = ""
OF_ref1 = ""
OF_ref2 = ""

# optional: distances to evaluate for inclusion in the ensemble dataframe
distances_to_evaluate = {'OF_gate': (OF_ref1, OF_ref2), 
                         'IF_gate': (IF_ref1, IF_ref2)}

# optional: user speficied reference structures for rmsd (i.e. from experiment)
use_reference_structures = False
#user_specified_structures = [base_directory + "CTNS_3965d_weblogo.pdb"]

# CPU cores to use for multiprocessing
num_processes = 6                

# number of principal components to keep in PCA
n_pcs = 3                  

# monte-carlo and umbrella sampling settings
collective_variable = 'PC1' # collective_variable to bin
collective_variable_2 = 'PC2' # second collective_variable to bin
use_defined_end_states = False # if false, use structures with lowest and highest values of CV
end_states = ['8DKE_IF.pdb, 8DKI_OF.pdb']
mc_n_bins = 32 # total number of bins
mc_n_runs = 100 # total number of batches of runs
reproducible_seeds = False
fixed_endstates = False # whether to freeze the end-state structures

# system geometry into which to embed the window structures for US 
template_pdb_for_umbrella_sampling = (base_directory + 'template_D305p_smallbilayer.pdb')
resid_offset = 115 #115 # useful if mismatch between the residue numbering in reference pdbs # NEW: FOR WRITEOUT ONLY

    # template should be a pdb from e.g. an unbiased simulation you've run before
    # containing everything in the simualtion box, protein, lipids, water etc. 
    # if you have a working topolgy for this system, you can re-use it here

# default variables for plotting
plot_variable_1 = 'PC1'   
plot_variable_2 = 'PC2'
plot_variable_3 = 'IF_gate'

# END SETTINGS #

# surpress warnings
warnings.filterwarnings(action='ignore', module='matplotlib')
warnings.filterwarnings(action='ignore', module='mdanalysis')
warnings.filterwarnings('ignore')

# pandas settings
pd.set_option('display.max_colwidth', None)

os.environ["OMP_NUM_THREADS"] = str(num_processes)

if use_reference_structures:
    for ref_structure in user_specified_structures:
        if not os.path.exists(ref_structure):
            raise FileNotFoundError(f"Reference structure {ref_structure} not found")
        else:
            print(f"Using reference structure {ref_structure}")
            u = mda.Universe(ref_structure)
            u.atoms.residues.resids -= resid_offset

# Create a colormap
cmap = plt.cm.get_cmap('tab20c')
colors = cmap(np.linspace(0, 1, 10))  # Sample 10 colors from the colormap
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors) # Set the default color cycle


In [2]:
# Structure retrieval - user specific - do whatever you need here to get a dictionary of structures and their associated pLDDT values

# copy structures from outputs to ensemble
for msa in msa_depths:
    structures_msa = [f for f in os.listdir(base_directory + 'output_' + msa) if f.endswith('.pdb')]
    for structure in structures_msa:
        # copy all the pdb files with "_relaxed_" in the name to the structure directory
        if "_relaxed_" in structure:
            # get the rank of the strcuture (last thing before the file extension)
            rank = 'rank_' + structure.split('_')[structure.split('_').index('rank')+1]

            # check that the structure is not already in the structure directory
            if not os.path.exists(structure_directory + '/msa-' + msa + '_' + rank + '.pdb'):
                os.system('cp ' + base_directory + 'output_' + msa + '/' + structure + ' ' + structure_directory + '/msa-' + msa + '_' + rank + '.pdb')

# get list of all files in structure directory directory with the pdb extension
structures = [f for f in os.listdir(structure_directory) if f.endswith('.pdb')]
structures.sort()


In [ ]:
# Apply pLDDT filters and define an atom group for the PCA and RMSD calculations
import ensemblr
from ensemblr.smart_selector import generate_selection_token
# reload the module to apply changes
importlib.reload(ensemblr.smart_selector)
from ensemblr.smart_selector import generate_selection_token

# copy file called "log.txt" from each output directory to the structure directory and name it after the msa
for msa in msa_depths:
    # logfile is a file in directory output_msa called log.txt
    structures_msa = [f for f in os.listdir(base_directory + 'output_' + msa) if f.endswith('log.txt')]
    for logfile in structures_msa:
            os.system('cp ' + base_directory + 'output_' + msa + '/' + logfile + ' ' + structure_directory + '/' + msa + '_log.txt')

# associate structures with pLDDT values
structure_and_pLDDT = {}
for msa in msa_depths:
    structures_msa = [f for f in os.listdir(structure_directory) if f.endswith('.pdb') and msa in f and str(msa) in f]
    # look up the corresponding log file
    log_file = structure_directory + msa + "_log.txt"   
    for structure in structures_msa:
        # get the rank of the structure
        rank = 'rank_' + structure.split('_')[structure.split('_').index('rank')+1] 
        rank = rank.split('.')[0] # remove the file extension
        with open(log_file, "r") as file:
            # match the rank with the pLDDT value
            for line in file:
                if rank in line:
                    pLDDT = line.split()[3].replace('pLDDT=','')
                    structure_and_pLDDT[structure] = float(pLDDT)
                    break

print('# Structures:', len(structure_and_pLDDT))

# reference pdb is the top ranked structure by pLDDT - this is used for SS assignment
reference_pdb = structure_directory + sorted(structure_and_pLDDT, key=structure_and_pLDDT.get, reverse=True)[0]
u = mda.Universe(reference_pdb) # reference universe
# renumber the reference pdb by the offset so resids are consistent with what hte user expects

# generate the selection token used for RMSD and PCA calculations
rmsd_selection = generate_selection_token(reference_pdb, conserved_residues=conserved_residues, excluded_residues=ignored_residues, explicitly_include=always_include)
mda.Universe(reference_pdb).select_atoms(rmsd_selection).write(base_directory + 'rmsd_selection.pdb') # write out the selection

print(rmsd_selection)

# main ensemble dataframe declared here
ensemble_df = pd.DataFrame(index=structure_and_pLDDT.keys())
ensemble_df.index.names = ['structure']

pLDDT_scores = {}
plddt_df = pd.DataFrame(list(structure_and_pLDDT.items()), columns=['structure', 'pDDLDT'])

# Merge the two DataFrames on the structure column
ensemble_df = pd.merge(ensemble_df, plddt_df, on='structure')

# filter by pLDDT
ensemble_df = ensemble_df[ensemble_df['pDDLDT'] > thresh_pLDDT]      # discard < thresh_pLDDT
structures = list(ensemble_df['structure'])                          # apply the filter to the list of structure names

print('# Structures (post-filtering):', len(ensemble_df))

# deal with the ref structures
if use_reference_structures:
    # add entries for the user-specified structures with pLDDT scores of 100
    for ref_structure in user_specified_structures:
        pLDDT_scores[ref_structure] = 100
    ref_structures = user_specified_structures
# if no reference structures, choose hightest pLDDT structure as a dummy reference
if not use_reference_structures:
    ref_structures = sorted(structure_and_pLDDT, key=structure_and_pLDDT.get, reverse=True)[:1]
print('Reference structure:', ref_structures)


In [ ]:
# deal with diff numbers of atoms

# for each structure, get the number of atoms
n_atoms = {}
for structure in structures:
    u = mda.Universe(structure_directory + structure)
    n_atoms[structure] = len(u.atoms)

# add the number of atoms to the ensemble dataframe
ensemble_df['n_atoms'] = ensemble_df['structure'].map(n_atoms)

# print the unique atom counts
print('Unique atom counts:', set(n_atoms.values()))

highest_atom_count = max(n_atoms.values())

# print the number of structures per atom count
print('Structures per atom count:')
print(ensemble_df['n_atoms'].value_counts())

# print the highest pLDDT structure per atom count (and the corresponding structure)
print('Highest pLDDT structure per atom count:')
print(ensemble_df.loc[ensemble_df.groupby('n_atoms')['pDDLDT'].idxmax()])

# drop structures with fewer atoms than the highest atom count from the dataframe
ensemble_df = ensemble_df[ensemble_df['n_atoms'] == highest_atom_count]

print('# Structures (post-filtering):', len(ensemble_df))

In [ ]:
# Run PCA on the ensemble

# write a multistate pdb file for the whole ensemble so MDA will interpret it as a trajectory
with mda.Writer(base_directory + 'ensemble.pdb', u.atoms.n_atoms) as W:
    for structure in ensemble_df['structure']:
        u = mda.Universe(structure_directory + structure, 
                         structure_directory + structure)
        u.atoms.segments.segids = 'A'
        u.atoms.chainIDs = 'A'
        W.write(u.select_atoms('all')) # this used to be rmsd selection but all should be correct and pca should still be done on the selection
 
# make a universe containing all the structures
u = mda.Universe(base_directory + "ensemble.pdb")

# align the ensemble to the selection token (and first structure)
aligner = mda.analysis.align.AlignTraj(u, u, select=rmsd_selection, in_memory=True).run()

# center the ensemble at the origin
u.atoms.translate(-u.atoms.center_of_mass())

# write out the aligned ensemble
with mda.Writer(base_directory + 'ensemble.pdb', u.atoms.n_atoms) as W:
    for ts in u.trajectory:
        W.write(u.atoms)

# perform principal component analysis
pc = PCA(u, select=rmsd_selection, align=True, mean=None, n_components=None).run()

# project coorindates onto the principal components
pc_projection = pc.transform(u.select_atoms(rmsd_selection), n_components=n_pcs)

# make a dataframe to store the principal component projections
pca_df = pd.DataFrame(pc_projection, columns=['PC{}'.format(i+1) for i in range(n_pcs)])
pca_df['structure'] = ensemble_df.index

# print out the PCs
display(pd.DataFrame(pca_df).head())

# show table of variances explained by each PC
display(pd.DataFrame((pc.cumulated_variance*100).round(), columns=['PC cumulated variance']).head())

#drop the structure column for plotting
pca_df_pairgrid = pca_df.drop('structure', axis=1)
g = sns.PairGrid(pca_df_pairgrid)
g.map(plt.scatter, marker='.')
plt.show()
pca_df_pairgrid = None

# add principal components to the dataframe - making sure the structures are in the same order
pca_df = pca_df.sort_values(by=['structure'])
pca_df = pca_df.reset_index(drop=True)
ensemble_df['PC1'] = pca_df['PC1'].values
ensemble_df['PC2'] = pca_df['PC2'].values
ensemble_df['PC3'] = pca_df['PC3'].values

# visualisation
n_pcs = 3
for i in range(n_pcs):
    pc_n =    pc.p_components[:, i]
    trans_n =   pc_projection[:, i]
    projected = np.outer(trans_n, pc_n) + pc.mean.flatten()
    coordinates = projected.reshape(len(trans_n), -1, 3)
    
    proj_n = mda.Merge(u.select_atoms(rmsd_selection))
    proj_n.load_new(coordinates)
    
    # write this to a multistate pdb file
    with mda.Writer(base_directory + 'pca{}.pdb'.format(i+1), proj_n.atoms.n_atoms) as W:
        for ts in proj_n.trajectory:
            W.write(proj_n.atoms)

In [ ]:
# Select reference structures to compare the ensemble to

from sklearn.cluster import HDBSCAN

# clustering 

# min samples is 10 percent of size of ensemble
clustering_min_samples = int(len(ensemble_df) * 0.1)

cluster = HDBSCAN(min_samples=clustering_min_samples, store_centers="medoid").fit(ensemble_df[['PC1', 'PC2', 'PC3']])

# add the cluster labels to the dataframe
ensemble_df['cluster'] = cluster.labels_

# get the number of clusters that are not noise (-1)
num_clusters = len(set(cluster.labels_)) - (1 if -1 in cluster.labels_ else 0)

cluster_representatives = []
for i in cluster.medoids_:
    # lookup the relevant structures in the dataframe (using medoid guarantees this has been sampled)
    cluster_representatives.append(ensemble_df[(ensemble_df['PC1'] == i[0]) & (ensemble_df['PC2'] == i[1]) & (ensemble_df['PC3'] == i[2])]['structure'].values[0])

cluster_best_pLDDT = []
# for each cluster number, get the structure with the highest pLDDT
for i in range(num_clusters):
    cluster_best_pLDDT.append(ensemble_df[ensemble_df['cluster'] == i].sort_values(by=['pDDLDT'], ascending=False)['structure'].values[0])

# display the best structures per cluster 
print('Top pLDDT per cluster:')
display(ensemble_df[ensemble_df['structure'].isin(cluster_best_pLDDT)].sort_values(by=['pDDLDT'], ascending=False))

# set these to the ref_structures if not using a user specified set of reference structures
if not use_reference_structures:
    ref_structures = cluster_representatives

# 3D plot of the clusters
xaxis = 'PC1'
yaxis = 'PC2'
zaxis = 'PC3'
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(ensemble_df[xaxis], ensemble_df[yaxis], ensemble_df[zaxis], c=ensemble_df['cluster'], cmap='Paired', s=1, alpha=1)
for i in cluster_representatives:
    ax.scatter(ensemble_df[(ensemble_df['structure'] == i)][xaxis], ensemble_df[(ensemble_df['structure'] == i)][yaxis], ensemble_df[(ensemble_df['structure'] == i)][zaxis], c='black', marker='x', s=200)
ax.set_xlabel(xaxis)
ax.set_ylabel(yaxis)
ax.set_zlabel(zaxis)
plt.show()

In [ ]:
# Populate the ensemble dataframe with properties

from ensemblr.misc_functions import get_rmsd_to_ref, get_sasa, calc_distance

# get rmsds with respect ot each of the reference structures (from PCA)
for ref_structure in ref_structures:
    rmsds = {}
    with Pool(processes=num_processes) as pool:
        rmsds = list(
            tqdm(
                pool.starmap(
                    get_rmsd_to_ref, 
                    [(structure_directory + structure, structure_directory + ref_structure, rmsd_selection) for structure in structures]
                ), 
                total=len(structures)
            )
        )
    ensemble_df['rmsd_to_' + ref_structure] = dict(rmsds).values()

sasas = {}
with Pool(processes=num_processes) as pool:
    sasas = list(tqdm(pool.imap(get_sasa, [(structure_directory + structure) for structure in structures]), total=len(structures)))
ensemble_df['sasa'] = dict(sasas).values()

if not distances_to_evaluate:
    print('No distances to evaluate')
else:
    # Iterate over the items in the dictionary
    for distance_name, (selection1, selection2) in distances_to_evaluate.items():
        distances = {}
        for structure in structures:
            distances[structure] = calc_distance(structure=structure_directory + structure, selection1=selection1, selection2=selection2)

        # add the distances to the dataframe
        ensemble_df[distance_name] = distances.values()

display(ensemble_df)

In [ ]:
# Make plots of ensemble properties

# display tables of closest to reference structures, lowest sasa, and most different
print('Represnetative structures from each cluster:')
for i in ref_structures:
    display(ensemble_df.sort_values(by=['rmsd_to_' + i]).head(1)[['structure', 'pDDLDT', 'PC1', 'PC2']])
print('Lowest 3 SASA:')
display(ensemble_df.sort_values(by=['sasa']).head(3)[['structure', 'cluster', 'pDDLDT', 'PC1', 'PC2']])

# plot ensemble
plt.hexbin(ensemble_df[plot_variable_1], ensemble_df[plot_variable_2], cmap='plasma', gridsize=50, mincnt=1, bins='log')
plt.xlabel(plot_variable_1)
plt.ylabel(plot_variable_2)
plt.colorbar(label='counts')
plt.title('Density of structures')

# annotate reference structures
for i in ref_structures:
    plt.scatter(ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_1], 
                ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_2], 
                c='black', marker='x', s=100)
    plt.annotate(i, (ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_1], 
                     ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_2]))
plt.show()

# for the distances_to_evaluate, plot histograms and map the distances onto the ensembles with hexbins
if not distances_to_evaluate:

    print('No distances to plot') 

else:

    print('Plotting distances')
    for distance_name in distances_to_evaluate.keys():
        plt.hexbin(ensemble_df[plot_variable_1], 
                   ensemble_df[plot_variable_2], 
                   C=ensemble_df[distance_name], 
                   cmap='coolwarm', gridsize=50, mincnt=1)
        plt.xlabel(plot_variable_1)
        plt.ylabel(plot_variable_2)
        plt.colorbar(label=distance_name)
        plt.title('Distance distrubution of {}'.format(distance_name))
        plt.show()

    print('Clustering on distances')
    clustering_min_samples = int(len(ensemble_df) * 0.01)
    cluster = HDBSCAN(min_samples=clustering_min_samples, store_centers="medoid").fit(ensemble_df[[distance_name for distance_name in distances_to_evaluate.keys()]])
    ensemble_df['cluster_dists'] = cluster.labels_
    num_clusters = len(set(cluster.labels_)) - (1 if -1 in cluster.labels_ else 0)
    print('Number of clusters:', num_clusters)
    # get highest pLDDT structure from each cluster
    # for each cluster_dists label, get the structure with the highest pLDDT
    cluster_dists_representatives = []
    for i in range(num_clusters):
        cluster_dists_representatives.append(ensemble_df[ensemble_df['cluster_dists'] == i].sort_values(by=['pDDLDT'], ascending=False)['structure'].values[0])
    print('Top pLDDT per cluster:')
    #display(ensemble_df[ensemble_df['structure'].isin(cluster_dists_representatives)].sort_values(by=['pDDLDT'], ascending=False))
    # hexbin with distance based clusters
    plt.hexbin(ensemble_df[plot_variable_1], ensemble_df[plot_variable_2], C=ensemble_df['cluster_dists'], reduce_C_function=np.amax, cmap='Blues', gridsize=50)
    # label the cluster representatives
    for i in cluster_dists_representatives:
        plt.scatter(ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_1], 
                    ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_2], 
                    c='black', marker='x', s=100)
        plt.annotate(i, (ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_1], 
                         ensemble_df.loc[ensemble_df['structure'] == i][plot_variable_2]))
    plt.show()

print('general ensemble properties')
# for each reference structure, plot the histogram of rmsd to that structure
for i in ref_structures:
    plt.hist(ensemble_df['rmsd_to_' + i], bins=20, alpha=0.5, label=i)
plt.legend(loc='upper right')
plt.xlabel('RMSD')
plt.ylabel('Count')
plt.xlim(0)
fig = plt.gcf()
fig.set_size_inches(10, 2)
plt.show()

# plot 2 pane histogram of SASA and pLDDT scores
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.hist(ensemble_df['sasa'], bins=20, alpha=0.5, label='sasa', color='C3')
ax1.set_xlabel('SASA')
ax1.set_ylabel('Count')
ax2.hist(ensemble_df['pDDLDT'], bins=20, alpha=0.5, label='pDDLDT')
ax2.set_xlabel('pDDLDT')
fig = plt.gcf()
fig.set_size_inches(10, 2)
plt.show()


In [ ]:
# Calculate property matrices (RMSD and CV) for the MC energy function

from ensemblr.calc_matrices import pdb_to_coordinates, get_rmsdmat_jax, get_diffmat

# test jax
import jax
print(jax.devices())

rmsd_matrix = get_rmsdmat_jax(pdb_to_coordinates(base_directory + 'ensemble.pdb'), ensemble_df)
cv_matrix = get_diffmat(ensemble_df, collective_variable)

if collective_variable_2 != None:
    cv2_matrix = get_diffmat(ensemble_df, collective_variable_2)

# show for inspection
fig, axes = plt.subplots(1, 2)
matrices = [cv_matrix, rmsd_matrix]
titles = ['CV matrix', 'RMSD matrix']

for ax, matrix, title in zip(axes, matrices, titles):
    ax.imshow(matrix, cmap='inferno')
    ax.set_title(title)

# show cv2 matrix if it exists
if collective_variable_2 != None:
    fig, ax = plt.subplots()
    ax.imshow(cv2_matrix, cmap='inferno')
    ax.set_title('CV2 matrix')



In [ ]:
# Assign bins for the MC path optimisation

#if use_defined_end_states == True: # NOT TESTED 
#
#    # get the values of the CV for these structures
#    max_value = ensemble_df[ensemble_df['structure'] == end_states[0]][collective_variable].values[0]
#    min_value = ensemble_df[ensemble_df['structure'] == end_states[1]][collective_variable].values[0]
#
#else:
#    
#    # just get the min and max value of the end states globally, like before
#    max_value = ensemble_df[collective_variable].max()
#    min_value = ensemble_df[collective_variable].min()

### get the ideal collective variable values per window between min and max_value using linear interpolation
#ideal_window_values = np.zeros(mc_n_bins)
#for i in range(mc_n_bins):
#    ideal_window_values[i] = np.interp(i, [0, mc_n_bins-1], [max_value, min_value])
#ideal_initial_path = []
#for i in ideal_window_values:
#    ideal_initial_path.append(ensemble_df[collective_variable].sub(i).abs().idxmin())
#
## replace the first and last structures in the ideal path with the end states
#if use_defined_end_states == True:
#    ideal_initial_path[0] = ensemble_df[ensemble_df['structure'] == end_state_1].index[0]
#    ideal_initial_path[-1] = ensemble_df[ensemble_df['structure'] == end_state_2].index[0]

## set up the bins for the binned path
#ensemble_df['closest_ideal_structure'] = ensemble_df[collective_variable].apply(lambda x: ideal_initial_path[np.argmin(np.abs(ideal_window_values - x))])
#ensemble_df['bin'] = ensemble_df['closest_ideal_structure'].apply(lambda x: ideal_initial_path.index(x))

# k means cluster the ensemble into mc_n_bins
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=mc_n_bins, random_state=1).fit(ensemble_df[[collective_variable]])
ensemble_df['bin'] = kmeans.labels_

# initial window values from the mean value of the cv in each bin/cluster
guess_window_values = np.sort(ensemble_df.groupby('bin')[collective_variable].mean().values)
ideal_initial_path = []
for i in guess_window_values:
    ideal_initial_path.append(ensemble_df[collective_variable].sub(i).abs().idxmin())

# ideal window values are updated once the MC path is optimised to linearly interpolate between the values of the end states

# plotting
plt.hist(ensemble_df['bin'], bins=mc_n_bins)
plt.show()

# show bin distribution
plt.hexbin(ensemble_df[plot_variable_1], ensemble_df[plot_variable_2], C=ensemble_df['bin'], cmap='tab20c', reduce_C_function=np.amax)
plt.colorbar()

# plot the initial guesses
for i in range(mc_n_bins):
    plt.scatter(ensemble_df.loc[ensemble_df.index == ideal_initial_path[i]][plot_variable_1], ensemble_df.loc[ensemble_df.index == ideal_initial_path[i]][plot_variable_2], marker='x', color='black', s=10)
for i in range(mc_n_bins):
    plt.annotate(i, (ensemble_df.loc[ensemble_df.index == ideal_initial_path[i]][plot_variable_1], ensemble_df.loc[ensemble_df.index == ideal_initial_path[i]][plot_variable_2]))

plt.show()


In [ ]:
# MC path optimisation, uses Pool for parallel execution
import ensemblr
from ensemblr.monte_carlo import mc_path_optimisation
# reload the module to ensure the latest version is used
importlib.reload(ensemblr.monte_carlo)

from ensemblr.monte_carlo import mc_path_optimisation

# run the monte carlo path optimisation
if reproducible_seeds:
    seeds = range(mc_n_runs)
else:
    seeds = np.random.randint(low=0, high=1e3, size=mc_n_runs)
with Pool(processes=num_processes) as pool:
    mc_runs = list(
        tqdm(
            pool.starmap(
                # positional arguments passed as a tuple
                mc_path_optimisation,
                zip(
                    seeds, 
                    [ideal_initial_path]*mc_n_runs, 
                    [rmsd_matrix]*mc_n_runs, 
                    [cv_matrix]*mc_n_runs, 
                    [ensemble_df]*mc_n_runs,
                    [str(fixed_endstates)]*mc_n_runs,
                    [cv2_matrix]*mc_n_runs,
                )
            ), 
            total=mc_n_runs
        )
    )
mc_runs_df = pd.DataFrame(mc_runs, columns=['energy', 'path', 'path structures', 'relaxation energies']).sort_values(by=['energy'])

# plotting below
def lookup_and_plot_energy_per_structure(mc_df, index, label, wf_rmsd=1, wf_cv=1): # Be warned this only uses the same energy function path optimisation if you set wf_rmsd and wf_cv correctly
    energy = {}
    path = mc_df.iloc[index]['path']
    for i in range(mc_n_bins - 1):
        energy[i] = wf_rmsd * np.sqrt(rmsd_matrix[np.where(ensemble_df.index.values == path[i])[0][0], np.where(ensemble_df.index.values == path[i+1])[0][0]]**2)
        energy[i] = wf_cv * np.sqrt(cv_matrix[np.where(ensemble_df.index.values == path[i])[0][0], np.where(ensemble_df.index.values == path[i+1])[0][0]]**2)
    plt.plot(energy.values(), label=label)
    plt.xlabel('Structure in path')
    plt.ylabel('Node/window energy')
    plt.legend()

lookup_and_plot_energy_per_structure(mc_runs_df, 0, 'best')
lookup_and_plot_energy_per_structure(mc_runs_df, -1, 'worst')
plt.show()
for i in range(1, mc_n_runs):
    plt.plot(mc_runs_df.iloc[i]['relaxation energies'])
plt.xlabel('MC step')
plt.ylabel('Path energy')
plt.show()

# write out the best run to a multistate pdb file for visualisation
with mda.Writer(base_directory + 'best_run.pdb', u.atoms.n_atoms) as W:
    # head gets the structures of the lowest energy path because the dataframe is sorted by energy above
    for structure in mc_runs_df.head(1)['path structures'].values[0]:   
        if structure == 'ref_outward.pdb' or structure == 'ref_inward.pdb':
            # useful if the reference structures have different metadata labelling (e.g. if they came from a pdb)
            u.atoms.segments.segids = 'A'
            u.atoms.chainIDs = 'A'
        u = mda.Universe(structure_directory + structure, structure_directory + structure)
        u.atoms.residues.resids += resid_offset
        W.write(u.select_atoms('protein'))


In [ ]:
# Plot and assess the optimal path

# get values of the cv for each window in the optimal path
actual_window_values = np.zeros(mc_n_bins)
for i in range(mc_n_bins):
    actual_window_values[i] = ensemble_df.loc[mc_runs_df.head(1)['path'].values[0][i]][collective_variable]

# update ideal window values to interpolate between end points of the actual window values (so US CV windows are evenly spaced)
ideal_window_values = np.interp(np.linspace(0, mc_n_bins-1, mc_n_bins), [0, mc_n_bins-1], [actual_window_values[0], actual_window_values[-1]])

# plotting
plt.hexbin(ensemble_df[plot_variable_1], ensemble_df[plot_variable_2], C=ensemble_df['bin'], cmap='tab20c', reduce_C_function=np.amax)
for structure in mc_runs_df.head(1)['path'].values[0]:
    plt.scatter(ensemble_df.loc[structure][plot_variable_1], ensemble_df.loc[structure][plot_variable_2], c='black', s=32)
# draw connecting lines
for i in range(len(mc_runs_df.head(1)['path'].values[0]) - 1):
    x1 = ensemble_df.loc[mc_runs_df.head(1)['path'].values[0][i]][plot_variable_1]
    y1 = ensemble_df.loc[mc_runs_df.head(1)['path'].values[0][i]][plot_variable_2]
    x2 = ensemble_df.loc[mc_runs_df.head(1)['path'].values[0][i+1]][plot_variable_1]
    y2 = ensemble_df.loc[mc_runs_df.head(1)['path'].values[0][i+1]][plot_variable_2]
    plt.plot([x1, x2], [y1, y2], c='black', alpha=0.6, linestyle='dashed')
# plot the initial guesses
for i in range(mc_n_bins):
    plt.scatter(ensemble_df.loc[ensemble_df.index == ideal_initial_path[i]][plot_variable_1], ensemble_df.loc[ensemble_df.index == ideal_initial_path[i]][plot_variable_2], marker='x', color='black', s=10)

plt.xlabel(plot_variable_1)
plt.ylabel(plot_variable_2)
plt.show()

fig, axs = plt.subplots(1, 2)
fig.set_size_inches(12, 4)

# Umbrella sampling quality control plots
axs[0].plot(ideal_window_values, label='ideal', color='black', alpha=0.2)
axs[0].scatter(range(mc_n_bins), ideal_window_values, c=range(mc_n_bins), cmap='coolwarm', alpha=0.5)



axs[0].plot(actual_window_values, label='actual', c='black', alpha=0.8)
axs[0].scatter(range(mc_n_bins), actual_window_values, c=range(mc_n_bins), cmap='coolwarm')
for i in range(mc_n_bins):
    axs[0].plot([i, i], [ideal_window_values[i], actual_window_values[i]], c='black', alpha=0.2, linestyle='dashed')
axs[0].set_xticks(range(mc_n_bins))
axs[0].set_xlabel('Window Number')
axs[0].set_ylabel(collective_variable + ' CV')
axs[0].legend()
axs[0].set_title('Ideal vs Actual Window CV Values')
offsets = np.abs(ideal_window_values - actual_window_values)
axs[1].bar(range(mc_n_bins), offsets, color='black', alpha=0.2)
axs[1].set_xlabel('Window Number')
axs[1].set_ylabel('Absolute Offset')
axs[1].set_title('Offsets from ideal window values')
plt.show()

# display actual vs idea in table
#print('Actual vs ideal window values:')
#window_values = pd.DataFrame({'ideal': ideal_window_values, 'actual': actual_window_values})
#display(window_values)


In [ ]:
# Set up umbrella sampling windows from the optimal path

# imports for now (to be moved later)
import ensemblr.us_window_setup
importlib.reload(ensemblr.us_window_setup)
from ensemblr.us_window_setup import align_universe, protein_coordinate_replacer, write_pca_ref_pdb, plumed_input_writer, fix_overlapping_atoms, prevent_threaded_lipids, fix_long_bonds, fix_overlapping_with_protein

# print a warning about this embedding process
print('Embedding structures into the template universe for umbrella sampling.\nWARNING: This process is not perfect! Check for threaded lipids.\nYou are welcome to use your own embedding method instead...')

window_aln_selection = 'name CA' #rmsd_selection + ' chainID A'

# template handling 
u_template = mda.Universe(template_pdb_for_umbrella_sampling, template_pdb_for_umbrella_sampling)

umbrella_sampling_directory = base_directory + 'umbrella_sampling/'

# get average coords for mda PC calculation and to use as a reference pdb for parsing MDA selections - PROBLEM WITH THIS 
u_average = mda.Universe(base_directory + 'ensemble.pdb')
u_average.trajectory[0]

u_average.select_atoms(rmsd_selection).positions = pc.mean
u_average.atoms.write(base_directory + 'average_structure.pdb')

## US window handler
window_directories = [] # prepare directories for each window
for i in range(mc_n_bins):
    window_directories.append(umbrella_sampling_directory + 'window_' + str(i))
    if not os.path.exists(window_directories[i]):
        os.makedirs(window_directories[i])
    shutil.copy(structure_directory + mc_runs_df.head(1)['path structures'].values[0][i], window_directories[i])    # copy the relevant structure in mc_runs_df to the window directory

# for each window, replace the coordinates of the protein in the template universe with the coordinates of the structure at the corresponding index in the best path
for i in range(mc_n_bins):
    u = u_template.copy()   
    structure = mc_runs_df.head(1)['path structures'].values[0][i] # make a universe out of the structure corresponding to the index in the best path
    u_structure = mda.Universe(structure_directory + structure, structure_directory + structure)
    u_structure = align_universe(u_structure, u, window_aln_selection) #align_universe(u_structure, u, rmsd_selection + ' and chainID A') # align the structures to the template universe

    # print window num
    print('Window:', i)

    # replace the coordinates of the protein in the template universe with the coordinates of the structure at the corresponding index in the best path
    protein_coordinate_replacer(u_structure, u, resid_offset=resid_offset, selection_token='protein and chainID A')
     
    # try to avoid and fix clashes
    prevent_threaded_lipids(u, selection_token='resname DLPC')
    fix_long_bonds(u, thresh=10.0, fudge_dist=1)
    fix_overlapping_atoms(u, selection_token='protein or (around 10 protein)', tolerance=0.2, step=0.1)

    # write out the universe to a pdb file
    u.atoms.write(window_directories[i] + '/window_' + str(i) + '.pdb')

    # store the first window as a reference structure
    ref_for_mda_selection_parsing = window_directories[0] + '/window_0.pdb'

## plumed CV handler
print('collective variable is:', collective_variable, '\n')

# if cv is a label in the distances to evaluate dictionary
if collective_variable in distances_to_evaluate.keys():

    pcavar = False

    # coms from the list specified at the beginning
    com1 = distances_to_evaluate[collective_variable][0]
    com2 = distances_to_evaluate[collective_variable][1]

    cv_values_for_plumed_input = ideal_window_values

    plumed_input_writer(cv_window_values=ideal_window_values, 
                        cv_reference_pdb=ref_for_mda_selection_parsing,
                        molinfo_pdb=ref_for_mda_selection_parsing, 
                        COM_1=com1, 
                        COM_2=com2,
                        multidir=True,
                        output_file=umbrella_sampling_directory + '/plumed_multidir.dat')

    # for each window, write a non-mulitdir (serial) plumed file inside the window directory
    for i in range(mc_n_bins):
        # get the corresponding value of the CV for the window
        cv_value_for_plumed_input = ideal_window_values[i]
        plumed_input_writer(cv_window_values=cv_value_for_plumed_input, 
                            cv_reference_pdb=ref_for_mda_selection_parsing,
                            molinfo_pdb=ref_for_mda_selection_parsing, 
                            COM_1=com1, 
                            COM_2=com2,
                            multidir=False,
                            output_file=window_directories[i] + '/plumed.dat')

# condition for PCA based CV
elif collective_variable.startswith('PC') and collective_variable[2:].isdigit():

    pcavar = True
    print('PCA based CV')

    # get the eigenvector from the MDA pca object
    eigenvector = pc.p_components[:, 0].reshape(-1,3) # first principal component, features reshaped to get cartesian components

    num_atoms_in_pca_selection = len(eigenvector)
    print('Number of atoms in selection:', num_atoms_in_pca_selection)

    # write out pdbs of u_averand and u to debug
    #u_average.atoms.write(base_directory + 'u_average.pdb')
    #u.atoms.write(base_directory + 'u.pdb')

    # if eigenvector is the wrong shape, theres probably a mismatch between what the selection token matches in the template and ensemble universes
    print('Length of eigenvector: ', len(eigenvector))

    # write out a plumed reference file for the linear subspace projection 
    write_pca_ref_pdb(u_structure=u_average, u_template=u, 
                      eigenvector=eigenvector, 
                      offset=resid_offset,
                      pca_selection_token=rmsd_selection,
                      output_file=umbrella_sampling_directory + 'pca_ref.pdb')

    cv_values_for_plumed_input = ideal_window_values

    plumed_input_writer(cv_window_values=cv_values_for_plumed_input, 
                        cv_reference_pdb=umbrella_sampling_directory + 'pca_ref.pdb', 
                        molinfo_pdb=ref_for_mda_selection_parsing,
                        scale_factor=num_atoms_in_pca_selection,
                        multidir=True,
                        output_file=umbrella_sampling_directory + '/plumed_multidir.dat')

    # for each window, write a non-mulitdir (serial) plumed file inside the window directory
    for i in range(mc_n_bins):
        cv_value_for_plumed_input = cv_values_for_plumed_input[i]
        plumed_input_writer(cv_window_values=cv_value_for_plumed_input, 
                            cv_reference_pdb=umbrella_sampling_directory + '../pca_ref.pdb', 
                            molinfo_pdb=ref_for_mda_selection_parsing,
                            scale_factor=num_atoms_in_pca_selection,
                            multidir=False,
                            output_file=window_directories[i] + '/plumed.dat')

else:
    print('Unknown collective variable. Writing generic plumed file')
    com1 = '{your atoms here}'
    com2 = '{your atoms here}'


In [ ]:
# DEBUGGING cell for plumed and MDA CV values

# read the COLVAR_MULTI files using glob
import glob
colvar_files = glob.glob(umbrella_sampling_directory + 'window_*/COLVAR')
# sort numerically by window number (second last part of the path)
colvar_files.sort(key=lambda x: int(x.split('/')[-2].split('_')[1]))
#print(colvar_files)

factor=int(len(u.select_atoms(rmsd_selection)))

# read second column second line to get CV value from plumed
cv_values = []
for file in colvar_files:
    with open(file, 'r') as f:
        lines = f.readlines()
        cv_values.append(float(lines[1].split()[1]))


# print number of atoms in rmsd_selection
print('Number of atoms in rmsd_selection:', len(u.select_atoms(rmsd_selection)))

# print the magnitude of the eigenvector
print('Magnitude of eigenvector:', np.linalg.norm(eigenvector))


# scale the "actual window values"
plt.plot(actual_window_values, label='MDA_actual', alpha=0.5, color='red')

# plot the cv values for plumed input
plt.plot(cv_values_for_plumed_input, label='ideal', alpha=0.5, color='black')

# plot the CV values
#plt.plot(cv_values, label='PLUMED', alpha=0.5, color='black')

# plot the CV values
plt.plot(cv_values, label='PLUMED', alpha=0.5, color='blue')

# scale cv values by number of atoms in the selection
#cv_value_scaled = np.array(cv_values) * factor
# plot the CV values
#plt.plot(cv_value_scaled, label='PLUMED_scaled', alpha=0.5, color='blue')

plt.legend()
plt.show()
